In [1]:
%pip install "qiskit==2.2.3" "qiskit-machine-learning==0.9.0" "qiskit-optimization==0.7.0" "qiskit-algorithms==0.4.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 20.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=ec5f674bcf84d68d61b01cd3379caf1ddb984603e90ce9b06558131a8193dc9d
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [9]:
import sys
import importlib.metadata as metadata
import numpy as np
import scipy
import sklearn
import qiskit
import qiskit_machine_learning
import qiskit_optimization
import qiskit_algorithms

expected = {
    "qiskit": "2.2.3",
    "qiskit-machine-learning": "0.9.0",
    "qiskit-optimization": "0.7.0",
    "qiskit-algorithms": "0.4.0",
}
print("Python:", sys.version.split()[0])
for package in [*expected, "numpy", "scipy", "scikit-learn"]:
    installed = metadata.version(package)
    print(f"{package}: {installed}")
    if package in expected:
        assert installed == expected[package], (
            f"{package} 버전을 확인하고 설치 셀 실행 후 세션을 다시 시작하세요."
        )
print("패키지 import와 버전 확인 완료")

Python: 3.13.18
qiskit: 2.2.3
qiskit-machine-learning: 0.9.0
qiskit-optimization: 0.7.0
qiskit-algorithms: 0.4.0
numpy: 2.4.6
scipy: 1.16.3
scikit-learn: 1.6.1
패키지 import와 버전 확인 완료


In [11]:
import sys
import importlib.util
import subprocess

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "torch",
        "--index-url", "https://download.pytorch.org/whl/cpu",
    ])

import torch
print("PyTorch:", torch.__version__)
print("실습 장치: CPU")
probe = torch.tensor([1.0, 2.0], device="cpu")
assert torch.equal(probe + 1, torch.tensor([2.0, 3.0]))
print("PyTorch 텐서 연산 확인 완료")

PyTorch: 2.14.1+cpu
실습 장치: CPU
PyTorch 텐서 연산 확인 완료


In [10]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from qiskit_optimization import QuadraticProgram

from qiskit.primitives import StatevectorSampler
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

# 1. 합성 악성 코드 데이터 생성 ----------------------------------------------------------
torch.manual_seed(0)
torch.set_num_threads(1)
code_length, num_patterns = 16, 6
# 각 행은 길이 16의 고정 정상 특징 벡터입니다. 값은 0~1의 무차원 점수입니다.
backgrounds = np.array([
    [.1,.2,.3,.4,.1,.2,.3,.4,.1,.2,.3,.4,.1,.2,.3,.4],
    [.2,.1,.4,.3,.2,.1,.4,.3,.2,.1,.4,.3,.2,.1,.4,.3],
    [.3,.4,.1,.2,.3,.4,.1,.2,.3,.4,.1,.2,.3,.4,.1,.2],
    [.4,.3,.2,.1,.4,.3,.2,.1,.4,.3,.2,.1,.4,.3,.2,.1],
    [.15,.25,.35,.45,.15,.25,.35,.45,.15,.25,.35,.45,.15,.25,.35,.45],
    [.25,.15,.45,.35,.25,.15,.45,.35,.25,.15,.45,.35,.25,.15,.45,.35]
], dtype=np.float32)
virus_patterns = np.zeros((num_patterns, code_length), dtype=np.float32)
for i in range(num_patterns):
    virus_patterns[i, i*2:i*2+3] = 1.0

def make_dataset(rows):
    features, labels = [], []
    for background in rows:
        for i in range(num_patterns):
            features.extend([background.copy(), np.clip(background + virus_patterns[i], 0, 1)])
            labels.extend([0, 1])
    return np.array(features, dtype=np.float32), np.array(labels, dtype=np.float32)

# 배경 단위 분할: 시험용 배경은 학습에 포함하지 않습니다.
X_train, y_train = make_dataset(backgrounds[:4])
X_test, y_test = make_dataset(backgrounds[4:])

# 2. 딥러닝 바이러스 탐지 모델 ----------------------------------------------------------
class VirusNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

device = torch.device("cpu")
model = VirusNet(code_length).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

X_train_t = torch.tensor(X_train, device=device)
y_train_t = torch.tensor(y_train.reshape(-1, 1), device=device)
X_test_t = torch.tensor(X_test, device=device)
y_test_t = torch.tensor(y_test.reshape(-1, 1), device=device)

epochs = 30
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    preds = (model(X_test_t) >= 0.5).float()
    acc = (preds.eq(y_test_t).float().mean()).item()
print(f"테스트 정확도: {acc:.3f}")

# 3. 패턴별 악성 확률 계산 ----------------------------------------------------------
model.eval()
pattern_scores = []
with torch.no_grad():
    for i in range(num_patterns):
        p_vec = torch.tensor(virus_patterns[i].reshape(1, -1),
                             dtype=torch.float32, device=device)
        prob = model(p_vec).item()
        pattern_scores.append(prob)
pattern_scores = np.array(pattern_scores)
print("패턴별 악성 확률:", pattern_scores)

# 4. QUBO 구성: "가장 악성인 패턴 1개만 선택" ---------------------------------------

# QUBO 목적:
#   min  -sum_i s_i x_i + P (sum_i x_i - 1)^2
#   → 악성 확률 s_i가 클수록 선택(x_i=1) 유리
#   → (sum x_i - 1)^2 패널티로 정확히 1개 선택 유도

qp = QuadraticProgram()
for i in range(num_patterns):
    qp.binary_var(name=f"x{i}")

penalty = 2.0  # 선택 개수를 1개로 맞추기 위한 페널티 계수 (점수 범위보다 충분히 크게)

linear = {}
quadratic = {}

# 선형항:  -s_i x_i  + penalty * (sum x_i - 1)^2에서 나온 항까지 합친 결과
# (전개 결과: 계수 = -s_i - penalty)
for i in range(num_patterns):
    coeff = -pattern_scores[i] - penalty
    linear[f"x{i}"] = coeff

# 2차항: penalty * (sum x_i - 1)^2 전개 시 2 * penalty * x_i x_j 항이 생김
for i in range(num_patterns):
    for j in range(i + 1, num_patterns):
        quadratic[(f"x{i}", f"x{j}")] = 2.0 * penalty

qp.minimize(constant=penalty, linear=linear, quadratic=quadratic)

# 5. QUBO와 Ising 해밀토니안
from itertools import product
H, offset = qp.to_ising()
diagonal = np.real(np.diag(H.to_matrix()))
for k in range(2**num_patterns):
    bits = np.array([(k >> i) & 1 for i in range(num_patterns)])
    expected = -float(pattern_scores @ bits) + penalty * (bits.sum() - 1)**2
    assert np.isclose(qp.objective.evaluate(bits), expected)
    assert np.isclose(diagonal[k] + offset, expected)

qaoa = QAOA(sampler=StatevectorSampler(default_shots=4096, seed=0),
            optimizer=COBYLA(maxiter=160), reps=2,
            initial_point=np.array([0.2, 0.4, 0.6, 0.8]))
energy_scale = max(1.0, float(np.max(np.abs(diagonal))))
result = qaoa.compute_minimum_eigenvalue(H / energy_scale)
dist = {int(k, 2) if isinstance(k, str) else int(k): float(p)
        for k, p in result.eigenstate.items()}
# 최종 측정에서 관측된 비트열 중 QUBO 비용이 가장 작은 값을 선택합니다.
best = min((k for k in dist if dist[k] > 0),
           key=lambda k: qp.objective.evaluate(
               np.array([(k >> j) & 1 for j in range(num_patterns)])))
x = np.array([(best >> i) & 1 for i in range(num_patterns)])
selected_patterns = np.flatnonzero(x).tolist()
objective = -float(pattern_scores @ x)
exact = -float(pattern_scores.max())
feasible_rate = optimal_rate = 0.0
for k, probability in dist.items():
    bits = np.array([(k >> i) & 1 for i in range(num_patterns)])
    if bits.sum() == 1:
        feasible_rate += probability
        if np.isclose(-float(pattern_scores @ bits), exact, atol=1e-8, rtol=0):
            optimal_rate += probability
print('선택 비트 [패턴0,...,패턴5]:', x.tolist())
print('선택 패턴:', selected_patterns)
print('선택 패턴 점수:', pattern_scores[selected_patterns])
print('원래 목적함수:', objective)
print('정확히 1개 선택 충족:', bool(x.sum() == 1))
print('선택 QUBO 목적값:', qp.objective.evaluate(x))
print('정확 최적값:', exact)
print('정확 최적값과 차이:', objective-exact if x.sum() == 1 else '제약 위반')
print('기대 Ising 에너지:', (float(result.eigenvalue.real) * energy_scale))
print('기대 QUBO 목적값:', (float(result.eigenvalue.real) * energy_scale) + offset)
print('선택 비트 관측 비율:', dist[best])
print('실행 가능한 해 관측 비율:', feasible_rate)
print('최적해 관측 비율:', optimal_rate)
assert np.isclose(sum(dist.values()), 1.0)

Classical SVM Accuracy: 1.0
Quantum SVM Accuracy : 0.875
학습/시험 크기: (16, 2) (8, 2)
QSVM 분류 결과:
              precision    recall  f1-score   support

           0      1.000     0.750     0.857         4
           1      0.800     1.000     0.889         4

    accuracy                          0.875         8
   macro avg      0.900     0.875     0.873         8
weighted avg      0.900     0.875     0.873         8



입력은 길이 16의 특징 벡터이며 각 값은 0~1 범위의 무차원 점수입니다. 정상 배경 6개와 연속 3개 위치에 1을 갖는 패턴 6개를 고정값으로 정의합니다. 네 개 배경으로 학습 48건, 두 개의 별도 배경으로 시험 24건을 구성합니다. 각 배경에 대해 정상 벡터와 패턴 하나가 포함된 벡터를 짝으로 생성하므로 클래스 수는 같습니다. 동일한 정상 벡터가 반복되는 소규모 교육용 데이터이며, 실제 악성코드 탐지 성능은 별도 자료로 평가합니다.

VirusNet은 16개 입력을 32개, 16개 은닉 노드와 sigmoid 출력으로 처리합니다. 학습은 30회 수행하며 BCELoss와 Adam을 사용합니다. 출력은 모델의 악성 분류 점수입니다. 독립적인 확률 보정은 수행하지 않습니다. 시험 정확도와 각 패턴 단독 입력의 점수를 따로 출력합니다.

패턴 점수 → 선택 변수 → 정확히 1개 선택 제약 → QUBO → Ising 해밀토니안 → QAOA 회로 → 각도 최적화 → 측정 → 패턴 복원 순서로 처리합니다. QUBO는 선택 점수의 음수 합과 선택 개수 위반 페널티를 더합니다. P=2.0이고 점수는 0~1이므로 1개 선택의 전역 최적해가 보장됩니다. 상수항 P도 포함하여 수식과 출력 에너지를 일치시킵니다. QAOA의 근사 탐색에서는 제약을 위반하는 관측도 발생할 수 있습니다.

초기 6큐비트 상태의 64개 비트열은 동일한 확률을 가집니다. 비용 회로는 γ(감마)에 따라 상대 위상을 바꾸고, 믹서의 β(베타) 회전은 간섭으로 측정 확률을 변화시킵니다. reps=2에서 네 개 각도를 COBYLA로 최적화하며 매 평가에 4,096회 측정을 사용합니다. energy_scale은 목적함수의 크기를 조절하며 출력 에너지는 원래 단위로 복원합니다.

최종 측정에서 관측된 비트열 중 QUBO 비용이 가장 작은 값을 선택하여 패턴 번호와 점수를 출력합니다. 관측되지 않은 비트열은 후보에 포함하지 않습니다. 원래 목적값은 선택 점수 합의 음수입니다. 정확히 1개가 선택된 경우 정확 최적값은 가장 높은 패턴 점수의 음수이며, 두 값의 차이가 0이면 최적 선택입니다. 기대 에너지는 전체 측정 분포의 평균이고, 선택 비트의 관측 비율은 해당 결과가 나온 빈도입니다. 실행 가능한 해 비율과 최적해 비율을 함께 확인합니다. 이 실습의 기본 설정은 단일 패턴 선택이며, 패턴 사이의 상호작용은 추가 이차항으로 모델링할 수 있습니다.